# Mã AES (Advanced Encryption Standard)

## 1. Giới thiệu

AES (Advanced Encryption Standard) là một hệ mật mã hiện đại thuộc loại **mã khối (block cipher)**.  
Thuật toán được chuẩn hóa bởi NIST vào năm 2001.

AES được sử dụng rộng rãi trong:
- Bảo mật mạng (HTTPS, VPN)
- Lưu trữ dữ liệu
- Ứng dụng di động và hệ thống nhúng

Khác với các hệ mật cổ điển, AES hoạt động trên **khối dữ liệu 128-bit** và sử dụng nhiều vòng biến đổi phức tạp.

## 2. Cơ sở lý thuyết

##### AES hoạt động trên:
- Block size: 128 bit (16 bytes)
- Key size: 128, 192 hoặc 256 bit

##### Dữ liệu được biểu diễn dưới dạng ma trận 4×4 byte:

<table style="border-collapse: collapse; text-align: center; font-weight: 500;">
  <tr>
    <td style="border:1px solid black; padding:8px;">b0</td>
    <td style="border:1px solid black; padding:8px;">b4</td>
    <td style="border:1px solid black; padding:8px;">b8</td>
    <td style="border:1px solid black; padding:8px;">b12</td>
  </tr>
  <tr>
    <td style="border:1px solid black; padding:8px;">b1</td>
    <td style="border:1px solid black; padding:8px;">b5</td>
    <td style="border:1px solid black; padding:8px;">b9</td>
    <td style="border:1px solid black; padding:8px;">b13</td>
  </tr>
  <tr>
    <td style="border:1px solid black; padding:8px;">b2</td>
    <td style="border:1px solid black; padding:8px;">b6</td>
    <td style="border:1px solid black; padding:8px;">b10</td>
    <td style="border:1px solid black; padding:8px;">b14</td>
  </tr>
  <tr>
    <td style="border:1px solid black; padding:8px;">b3</td>
    <td style="border:1px solid black; padding:8px;">b7</td>
    <td style="border:1px solid black; padding:8px;">b11</td>
    <td style="border:1px solid black; padding:8px;">b15</td>
  </tr>
</table>

##### AES gồm các phép biến đổi chính:
- SubBytes (thay thế S-box)
- ShiftRows (dịch hàng)
- MixColumns (trộn cột)
- AddRoundKey (XOR với khóa)

<h2>3. Ý tưởng thuật toán AES</h2>

<h3>Quy trình mã hóa</h3>

<ol style="line-height:1.8;">
  <li><b>Chuyển plaintext</b> thành ma trận <code>state 4×4</code></li>
  <li><b>Sinh round key</b> từ khóa ban đầu (Key Expansion)</li>
  <li><b>Thực hiện các vòng mã hóa:</b>
    <ul>
      <li>SubBytes (thay thế S-box)</li>
      <li>ShiftRows (dịch hàng)</li>
      <li>MixColumns (trộn cột - bỏ ở vòng cuối)</li>
      <li>AddRoundKey (XOR với khóa)</li>
    </ul>
  </li>
  <li><b>Xuất ciphertext</b></li>
</ol>

<h3>Số vòng theo độ dài khóa</h3>

<table style="border-collapse:collapse;text-align:center;">
<tr>
  <th style="border:1px solid black;padding:8px;">Chuẩn AES</th>
  <th style="border:1px solid black;padding:8px;">Độ dài khóa</th>
  <th style="border:1px solid black;padding:8px;">Số vòng</th>
</tr>
<tr>
  <td style="border:1px solid black;padding:8px;">AES-128</td>
  <td style="border:1px solid black;padding:8px;">128 bit</td>
  <td style="border:1px solid black;padding:8px;">10</td>
</tr>
<tr>
  <td style="border:1px solid black;padding:8px;">AES-192</td>
  <td style="border:1px solid black;padding:8px;">192 bit</td>
  <td style="border:1px solid black;padding:8px;">12</td>
</tr>
<tr>
  <td style="border:1px solid black;padding:8px;">AES-256</td>
  <td style="border:1px solid black;padding:8px;">256 bit</td>
  <td style="border:1px solid black;padding:8px;">14</td>
</tr>
</table>


## 4. Hàm chuyển text (plaintext & key) thành ma trận

In [1]:
def text_to_bytes(text):
    return [ord(c) for c in text]
def pad_bytes(byte_list, size=16):
    return byte_list + [0]*(size - len(byte_list))
def bytes_to_state(byte_list):
    state = [[0]*4 for _ in range(4)]
    
    for i in range(16):
        row = i % 4
        col = i // 4
        state[row][col] = byte_list[i]
    
    return state
def text_to_state(text):
    b = text_to_bytes(text)
    b = pad_bytes(b)
    return bytes_to_state(b)

#### AES không làm việc trực tiếp với ký tự, mà nó làm việc với byte. Quy trình chuyển đổi gồm 2 bước như sau:
1. Chuyển mỗi ký tự sang mã ASCII (0–255)
2. Đảm bảo đủ 16 byte:
   - Nếu thiếu → padding (thêm 0x00)
   - Nếu dư → cắt
3. Điền vào ma trận 4×4 theo **column-major** (Điền theo cột, không phải theo hàng => Nếu điền theo hàng từ trái sang phải thì sẽ là sai)
   
<h3>Ví dụ</h3>

<table style="border-collapse:collapse;">
<tr>
  <td style="border:1px solid black;padding:8px;"><b>Plaintext</b></td>
  <td style="border:1px solid black;padding:8px;">strength</td>
</tr>
<tr>
  <td style="border:1px solid black;padding:8px;"><b>Key</b></td>
  <td style="border:1px solid black;padding:8px;">spirits</td>
</tr>
</table>

<h3>Chuyển plaintext "strength" thành byte (ASCII)</h3>

<table style="border-collapse:collapse;text-align:center;">
<tr>
  <th style="border:1px solid black;padding:6px;">Char</th>
  <th style="border:1px solid black;padding:6px;">ASCII (hex)</th>
</tr>
<tr><td style="border:1px solid black;padding:6px;">s</td><td style="border:1px solid black;padding:6px;">0x73</td></tr>
<tr><td style="border:1px solid black;padding:6px;">t</td><td style="border:1px solid black;padding:6px;">0x74</td></tr>
<tr><td style="border:1px solid black;padding:6px;">r</td><td style="border:1px solid black;padding:6px;">0x72</td></tr>
<tr><td style="border:1px solid black;padding:6px;">e</td><td style="border:1px solid black;padding:6px;">0x65</td></tr>
<tr><td style="border:1px solid black;padding:6px;">n</td><td style="border:1px solid black;padding:6px;">0x6e</td></tr>
<tr><td style="border:1px solid black;padding:6px;">g</td><td style="border:1px solid black;padding:6px;">0x67</td></tr>
<tr><td style="border:1px solid black;padding:6px;">t</td><td style="border:1px solid black;padding:6px;">0x74</td></tr>
<tr><td style="border:1px solid black;padding:6px;">h</td><td style="border:1px solid black;padding:6px;">0x68</td></tr>
</table>

<h3>Padding (thêm 0x00 cho đủ 16 byte)</h3>

<table style="border-collapse:collapse;text-align:center;">
<tr>
  <td style="border:1px solid black;padding:8px;">73</td>
  <td style="border:1px solid black;padding:8px;">74</td>
  <td style="border:1px solid black;padding:8px;">72</td>
  <td style="border:1px solid black;padding:8px;">65</td>
  <td style="border:1px solid black;padding:8px;">6e</td>
  <td style="border:1px solid black;padding:8px;">67</td>
  <td style="border:1px solid black;padding:8px;">74</td>
  <td style="border:1px solid black;padding:8px;">68</td>
  <td style="border:1px solid black;padding:8px;background:#eee;">00</td>
  <td style="border:1px solid black;padding:8px;background:#eee;">00</td>
  <td style="border:1px solid black;padding:8px;background:#eee;">00</td>
  <td style="border:1px solid black;padding:8px;background:#eee;">00</td>
  <td style="border:1px solid black;padding:8px;background:#eee;">00</td>
  <td style="border:1px solid black;padding:8px;background:#eee;">00</td>
  <td style="border:1px solid black;padding:8px;background:#eee;">00</td>
  <td style="border:1px solid black;padding:8px;background:#eee;">00</td>
</tr>
</table>

<h3>Ma trận plaintext</h3>

<table style="border-collapse:collapse;text-align:center;">
<tr>
  <td style="border:1px solid black;padding:8px;">73</td>
  <td style="border:1px solid black;padding:8px;">6e</td>
  <td style="border:1px solid black;padding:8px;">00</td>
  <td style="border:1px solid black;padding:8px;">00</td>
</tr>
<tr>
  <td style="border:1px solid black;padding:8px;">74</td>
  <td style="border:1px solid black;padding:8px;">67</td>
  <td style="border:1px solid black;padding:8px;">00</td>
  <td style="border:1px solid black;padding:8px;">00</td>
</tr>
<tr>
  <td style="border:1px solid black;padding:8px;">72</td>
  <td style="border:1px solid black;padding:8px;">74</td>
  <td style="border:1px solid black;padding:8px;">00</td>
  <td style="border:1px solid black;padding:8px;">00</td>
</tr>
<tr>
  <td style="border:1px solid black;padding:8px;">65</td>
  <td style="border:1px solid black;padding:8px;">68</td>
  <td style="border:1px solid black;padding:8px;">00</td>
  <td style="border:1px solid black;padding:8px;">00</td>
</tr>
</table>

<h3>Chuyển key "spirits" thành byte (ASCII)</h3>
<table style="border-collapse:collapse;text-align:center;">
<tr>
  <th style="border:1px solid black;padding:8px;">Char</th>
  <th style="border:1px solid black;padding:8px;">ASCII (hex)</th>
</tr>
<tr><td style="border:1px solid black;padding:8px;">s</td><td style="border:1px solid black;padding:8px;">0x73</td></tr>
<tr><td style="border:1px solid black;padding:8px;">p</td><td style="border:1px solid black;padding:8px;">0x70</td></tr>
<tr><td style="border:1px solid black;padding:8px;">i</td><td style="border:1px solid black;padding:8px;">0x69</td></tr>
<tr><td style="border:1px solid black;padding:8px;">r</td><td style="border:1px solid black;padding:8px;">0x72</td></tr>
<tr><td style="border:1px solid black;padding:8px;">i</td><td style="border:1px solid black;padding:8px;">0x69</td></tr>
<tr><td style="border:1px solid black;padding:8px;">t</td><td style="border:1px solid black;padding:8px;">0x74</td></tr>
<tr><td style="border:1px solid black;padding:8px;">s</td><td style="border:1px solid black;padding:8px;">0x73</td></tr>
</table>

<h3>Padding (thêm 0x00 cho đủ 16 byte)</h3>
<table style="border-collapse:collapse;text-align:center;">
<tr>
  <td style="border:1px solid black;padding:8px;">73</td>
  <td style="border:1px solid black;padding:8px;">70</td>
  <td style="border:1px solid black;padding:8px;">69</td>
  <td style="border:1px solid black;padding:8px;">72</td>
  <td style="border:1px solid black;padding:8px;">69</td>
  <td style="border:1px solid black;padding:8px;">74</td>
  <td style="border:1px solid black;padding:8px;">73</td>
  <td style="border:1px solid black;padding:8px;background:#eee;">00</td>
  <td style="border:1px solid black;padding:8px;background:#eee;">00</td>
  <td style="border:1px solid black;padding:8px;background:#eee;">00</td>
  <td style="border:1px solid black;padding:8px;background:#eee;">00</td>
  <td style="border:1px solid black;padding:8px;background:#eee;">00</td>
  <td style="border:1px solid black;padding:8px;background:#eee;">00</td>
  <td style="border:1px solid black;padding:8px;background:#eee;">00</td>
  <td style="border:1px solid black;padding:8px;background:#eee;">00</td>
  <td style="border:1px solid black;padding:8px;background:#eee;">00</td>
</tr>
</table>

<h3>Ma trận key</h3>

<table style="border-collapse:collapse;text-align:center;">
<tr>
  <td style="border:1px solid black;padding:8px;">73</td>
  <td style="border:1px solid black;padding:8px;">69</td>
  <td style="border:1px solid black;padding:8px;">00</td>
  <td style="border:1px solid black;padding:8px;">00</td>
</tr>
<tr>
  <td style="border:1px solid black;padding:8px;">70</td>
  <td style="border:1px solid black;padding:8px;">74</td>
  <td style="border:1px solid black;padding:8px;">00</td>
  <td style="border:1px solid black;padding:8px;">00</td>
</tr>
<tr>
  <td style="border:1px solid black;padding:8px;">69</td>
  <td style="border:1px solid black;padding:8px;">73</td>
  <td style="border:1px solid black;padding:8px;">00</td>
  <td style="border:1px solid black;padding:8px;">00</td>
</tr>
<tr>
  <td style="border:1px solid black;padding:8px;">72</td>
  <td style="border:1px solid black;padding:8px;">00</td>
  <td style="border:1px solid black;padding:8px;">00</td>
  <td style="border:1px solid black;padding:8px;">00</td>
</tr>
</table>

## 4. Bảng S-box

In [2]:
S_BOX = [
[0x63,0x7c,0x77,0x7b,0xf2,0x6b,0x6f,0xc5,0x30,0x01,0x67,0x2b,0xfe,0xd7,0xab,0x76],
[0xca,0x82,0xc9,0x7d,0xfa,0x59,0x47,0xf0,0xad,0xd4,0xa2,0xaf,0x9c,0xa4,0x72,0xc0],
[0xb7,0xfd,0x93,0x26,0x36,0x3f,0xf7,0xcc,0x34,0xa5,0xe5,0xf1,0x71,0xd8,0x31,0x15],
[0x04,0xc7,0x23,0xc3,0x18,0x96,0x05,0x9a,0x07,0x12,0x80,0xe2,0xeb,0x27,0xb2,0x75],
[0x09,0x83,0x2c,0x1a,0x1b,0x6e,0x5a,0xa0,0x52,0x3b,0xd6,0xb3,0x29,0xe3,0x2f,0x84],
[0x53,0xd1,0x00,0xed,0x20,0xfc,0xb1,0x5b,0x6a,0xcb,0xbe,0x39,0x4a,0x4c,0x58,0xcf],
[0xd0,0xef,0xaa,0xfb,0x43,0x4d,0x33,0x85,0x45,0xf9,0x02,0x7f,0x50,0x3c,0x9f,0xa8],
[0x51,0xa3,0x40,0x8f,0x92,0x9d,0x38,0xf5,0xbc,0xb6,0xda,0x21,0x10,0xff,0xf3,0xd2],
[0xcd,0x0c,0x13,0xec,0x5f,0x97,0x44,0x17,0xc4,0xa7,0x7e,0x3d,0x64,0x5d,0x19,0x73],
[0x60,0x81,0x4f,0xdc,0x22,0x2a,0x90,0x88,0x46,0xee,0xb8,0x14,0xde,0x5e,0x0b,0xdb],
[0xe0,0x32,0x3a,0x0a,0x49,0x06,0x24,0x5c,0xc2,0xd3,0xac,0x62,0x91,0x95,0xe4,0x79],
[0xe7,0xc8,0x37,0x6d,0x8d,0xd5,0x4e,0xa9,0x6c,0x56,0xf4,0xea,0x65,0x7a,0xae,0x08],
[0xba,0x78,0x25,0x2e,0x1c,0xa6,0xb4,0xc6,0xe8,0xdd,0x74,0x1f,0x4b,0xbd,0x8b,0x8a],
[0x70,0x3e,0xb5,0x66,0x48,0x03,0xf6,0x0e,0x61,0x35,0x57,0xb9,0x86,0xc1,0x1d,0x9e],
[0xe1,0xf8,0x98,0x11,0x69,0xd9,0x8e,0x94,0x9b,0x1e,0x87,0xe9,0xce,0x55,0x28,0xdf],
[0x8c,0xa1,0x89,0x0d,0xbf,0xe6,0x42,0x68,0x41,0x99,0x2d,0x0f,0xb0,0x54,0xbb,0x16]
]

S-box dùng để **thay thế phi tuyến**, giúp tăng độ an toàn của AES.

## 5. Hàm SubBytes

In [3]:
def sub_bytes(state):
    return [[S_BOX[b >> 4][b & 0x0F] for b in row] for row in state]

Hàm này thay mỗi byte trong state bằng giá trị tương ứng trong S-box.

## 6. Hàm ShiftRows

<h4>Trước ShiftRows</h4>
<table style="border-collapse: collapse; text-align:center;">
<tr>
  <td style="border:1px solid black;padding:8px;">00</td>
  <td style="border:1px solid black;padding:8px;">01</td>
  <td style="border:1px solid black;padding:8px;">02</td>
  <td style="border:1px solid black;padding:8px;">03</td>
</tr>
<tr>
  <td style="border:1px solid black;padding:8px;">10</td>
  <td style="border:1px solid black;padding:8px;">11</td>
  <td style="border:1px solid black;padding:8px;">12</td>
  <td style="border:1px solid black;padding:8px;">13</td>
</tr>
<tr>
  <td style="border:1px solid black;padding:8px;">20</td>
  <td style="border:1px solid black;padding:8px;">21</td>
  <td style="border:1px solid black;padding:8px;">22</td>
  <td style="border:1px solid black;padding:8px;">23</td>
</tr>
<tr>
  <td style="border:1px solid black;padding:8px;">30</td>
  <td style="border:1px solid black;padding:8px;">31</td>
  <td style="border:1px solid black;padding:8px;">32</td>
  <td style="border:1px solid black;padding:8px;">33</td>
</tr>
</table>

<h4>Sau ShiftRows</h4>
<table style="border-collapse: collapse; text-align:center;">
<tr>
  <td style="border:1px solid black;padding:8px;">00</td>
  <td style="border:1px solid black;padding:8px;">01</td>
  <td style="border:1px solid black;padding:8px;">02</td>
  <td style="border:1px solid black;padding:8px;">03</td>
</tr>
<tr>
  <td style="border:1px solid black;padding:8px;">11</td>
  <td style="border:1px solid black;padding:8px;">12</td>
  <td style="border:1px solid black;padding:8px;">13</td>
  <td style="border:1px solid black;padding:8px;">10</td>
</tr>
<tr>
  <td style="border:1px solid black;padding:8px;">22</td>
  <td style="border:1px solid black;padding:8px;">23</td>
  <td style="border:1px solid black;padding:8px;">20</td>
  <td style="border:1px solid black;padding:8px;">21</td>
</tr>
<tr>
  <td style="border:1px solid black;padding:8px;">33</td>
  <td style="border:1px solid black;padding:8px;">30</td>
  <td style="border:1px solid black;padding:8px;">31</td>
  <td style="border:1px solid black;padding:8px;">32</td>
</tr>
</table>

In [4]:
def shift_rows(state):
    return [
        state[0],
        state[1][1:] + state[1][:1],
        state[2][2:] + state[2][:2],
        state[3][3:] + state[3][:3],
    ]

##### Dịch vòng các hàng:
- Hàng 0: giữ nguyên
- Hàng 1: dịch trái 1
- Hàng 2: dịch trái 2
- Hàng 3: dịch trái 3

## 7. Nhân trong trường Galois 2^8 (giới hạn 2^8 phần tử)

In [5]:
def gmul(a, b):
    p = 0
    for _ in range(8):
        if b & 1:
            p ^= a
        hi_bit = a & 0x80
        a <<= 1
        if hi_bit:
            a ^= 0x1b
        b >>= 1
    return p % 256

<h3>Ví dụ: nhân 0x57 × 0x13 trong GF(2^8)</h3>

<table style="border-collapse:collapse;text-align:center;">
<tr><th style="border:1px solid black;padding:8px;">Bước</th><th style="border:1px solid black;padding:8px;">Giá trị</th></tr>
<tr><td style="border:1px solid black;padding:8px;">a</td><td style="border:1px solid black;padding:8px;">0x57</td></tr>
<tr><td style="border:1px solid black;padding:8px;">b</td><td style="border:1px solid black;padding:8px;">0x13</td></tr>
<tr><td style="border:1px solid black;padding:8px;">Kết quả</td><td style="border:1px solid black;padding:8px;">0xFE</td></tr>
</table>


Trong AES, các phép nhân không thực hiện trên số nguyên thông thường mà trên trường hữu hạn GF(2^8).

##### Cụ thể:
- Mỗi byte được xem như một đa thức bậc ≤ 7
- Phép nhân được thực hiện theo đa thức
- Sau đó lấy modulo đa thức bất khả quy: x⁸ + x⁴ + x³ + x + 1 (0x11B)

##### Ý nghĩa:
- Đảm bảo tính **phi tuyến và khó đảo ngược**
- Là nền tảng cho bước MixColumns

##### Điểm quan trọng:
- Không có phép cộng/trừ → chỉ dùng XOR
- Phép nhân dùng shift + XOR + giảm modulo

## 8. Hàm MixColumns

<h3>Ma trận MixColumns</h3>

<table style="border-collapse:collapse;text-align:center;">
<tr>
<td style="border:1px solid black;padding:10px;">2</td>
<td style="border:1px solid black;padding:10px;">3</td>
<td style="border:1px solid black;padding:10px;">1</td>
<td style="border:1px solid black;padding:10px;">1</td>
</tr>
<tr>
<td style="border:1px solid black;padding:10px;">1</td>
<td style="border:1px solid black;padding:10px;">2</td>
<td style="border:1px solid black;padding:10px;">3</td>
<td style="border:1px solid black;padding:10px;">1</td>
</tr>
<tr>
<td style="border:1px solid black;padding:10px;">1</td>
<td style="border:1px solid black;padding:10px;">1</td>
<td style="border:1px solid black;padding:10px;">2</td>
<td style="border:1px solid black;padding:10px;">3</td>
</tr>
<tr>
<td style="border:1px solid black;padding:10px;">3</td>
<td style="border:1px solid black;padding:10px;">1</td>
<td style="border:1px solid black;padding:10px;">1</td>
<td style="border:1px solid black;padding:10px;">2</td>
</tr>
</table>

<h3>Biến đổi một cột</h3>

<table style="border-collapse:collapse;text-align:center;">
<tr>
<td style="border:1px solid black;padding:10px;">a0</td>
<td style="border:1px solid black;padding:10px;">a1</td>
<td style="border:1px solid black;padding:10px;">a2</td>
<td style="border:1px solid black;padding:10px;">a3</td>
</tr>
</table>

<p>→ Sau MixColumns:</p>

<ul>
<li>b0 = 2·a0 ⊕ 3·a1 ⊕ a2 ⊕ a3</li>
<li>b1 = a0 ⊕ 2·a1 ⊕ 3·a2 ⊕ a3</li>
<li>b2 = a0 ⊕ a1 ⊕ 2·a2 ⊕ 3·a3</li>
<li>b3 = 3·a0 ⊕ a1 ⊕ a2 ⊕ 2·a3</li>
</ul>

In [6]:
def mix_single_column(a):
    return [
        gmul(a[0],2) ^ gmul(a[1],3) ^ a[2] ^ a[3],
        a[0] ^ gmul(a[1],2) ^ gmul(a[2],3) ^ a[3],
        a[0] ^ a[1] ^ gmul(a[2],2) ^ gmul(a[3],3),
        gmul(a[0],3) ^ a[1] ^ a[2] ^ gmul(a[3],2)
    ]

def mix_columns(state):
    return [mix_single_column(col) for col in state]

MixColumns là bước **khuếch tán (diffusion)** trong AES, nếu không có MixColumns, AES sẽ dễ bị phân tích theo từng byte riêng lẻ.
##### Cách hoạt động:
- Mỗi cột của state được xem như một vector 4 byte
- Nhân với một ma trận cố định trong GF(2^8)

##### Mục tiêu:
- Trộn thông tin giữa các byte trong cùng một cột
- Một byte đầu vào ảnh hưởng đến toàn bộ cột đầu ra

##### Tính chất quan trọng:
- Biến đổi tuyến tính (linear transformation)
- Có thể đảo ngược (để giải mã)

## 9. Hàm AddRoundKey

In [7]:
def add_round_key(state, key):
    return [[state[i][j] ^ key[i][j] for j in range(4)] for i in range(4)]

<h3>Phép XOR giữa State và Round Key</h3>

<table style="border-collapse:collapse;text-align:center;">
<tr>
<td style="border:1px solid black;padding:10px;">State</td>
<td style="border:1px solid black;padding:10px;">⊕</td>
<td style="border:1px solid black;padding:10px;">Key</td>
<td style="border:1px solid black;padding:10px;">=</td>
<td style="border:1px solid black;padding:10px;">Kết quả</td>
</tr>
<tr>
<td style="border:1px solid black;padding:10px;">0x32</td>
<td style="border:1px solid black;padding:10px;">⊕</td>
<td style="border:1px solid black;padding:10px;">0x2b</td>
<td style="border:1px solid black;padding:10px;">=</td>
<td style="border:1px solid black;padding:10px;">0x19</td>
</tr>
</table>

### Giải thích

AddRoundKey là bước kết hợp dữ liệu với khóa.

##### Cách thực hiện:
- XOR từng byte của state với byte tương ứng trong round key

##### Đặc điểm:
- Đây là bước duy nhất trực tiếp sử dụng khóa
- XOR là phép toán:
  - Nhanh
  - Dễ đảo ngược (XOR 2 lần sẽ trở về ban đầu)

##### Ý nghĩa:
- Đưa yếu tố bí mật (key) vào quá trình mã hóa
- Nếu không có bước này → AES không còn là hệ mật

##### Quan trọng:
- Tất cả các bước khác đều “công khai”
- Độ an toàn phụ thuộc hoàn toàn vào key ở bước này

## 10. Hàm KeyExpansion

In [8]:
RCON = [0x01,0x02,0x04,0x08,0x10,0x20,0x40,0x80,0x1B,0x36]

def rot_word(word):
    return word[1:] + word[:1]

def sub_word(word):
    return [S_BOX[b >> 4][b & 0x0F] for b in word]

def key_expansion(key):
    w = [list(col) for col in key]
    for i in range(4, 44):
        temp = w[i-1]
        if i % 4 == 0:
            temp = sub_word(rot_word(temp))
            temp[0] ^= RCON[i//4 - 1]
        w.append([w[i-4][j] ^ temp[j] for j in range(4)])
    return w

<h3>Quy trình tạo Round Key</h3>

<ol>
<li>Lấy 4 word ban đầu từ key</li>
<li>Áp dụng RotWord</li>
<li>Áp dụng SubWord (S-box)</li>
<li>XOR với Rcon</li>
<li>XOR với word trước đó</li>
</ol>

<h3>Minh họa RotWord</h3>

<table style="border-collapse:collapse;text-align:center;">
<tr>
<td style="border:1px solid black;padding:10px;">[a0 a1 a2 a3]</td>
<td style="border:1px solid black;padding:10px;">→</td>
<td style="border:1px solid black;padding:10px;">[a1 a2 a3 a0]</td>
</tr>
</table>

<h3>Minh họa SubWord</h3>

<table style="border-collapse:collapse;text-align:center;">
<tr>
<td style="border:1px solid black;padding:10px;">0x19</td>
<td style="border:1px solid black;padding:10px;">→</td>
<td style="border:1px solid black;padding:10px;">S_BOX[1][9]</td>
</tr>
</table>

## 11. Chạy thử chương trình

In [9]:
def aes_encrypt(plaintext, key):
    state = plaintext
    w = key_expansion(key)

    state = add_round_key(state, w[:4])

    for round in range(1,10):
        state = sub_bytes(state)
        state = shift_rows(state)
        state = mix_columns(state)
        state = add_round_key(state, w[4*round:4*(round+1)])

    state = sub_bytes(state)
    state = shift_rows(state)
    state = add_round_key(state, w[40:44])

    return state

In [10]:
from IPython.display import display, HTML

def show_state(state, title):
    html = f"<h3>{title}</h3><table style='border-collapse:collapse;'>"
    for row in state:
        html += "<tr>"
        for val in row:
            html += f"<td style='border:1px solid black;padding:10px;'>{hex(val)}</td>"
        html += "</tr>"
    html += "</table>"
    display(HTML(html))

def show_cipher_hex_ascii(state):
    html = "<h3>Ciphertext (Hex + ASCII)</h3>"
    hex_row = ""
    ascii_row = ""

    for col in range(4):
        for row in range(4):
            val = state[row][col]
            hex_row += f"<td style='border:1px solid black;padding:8px;'>{val:02x}</td>"

            if 32 <= val <= 126:
                ascii_row += f"<td style='border:1px solid black;padding:8px;'>{chr(val)}</td>"
            else:
                ascii_row += f"<td style='border:1px solid black;padding:8px;background:#eee;'>.</td>"

    html += f"""
    <table style='border-collapse:collapse;text-align:center;'>
        <tr><th colspan="16" style='border:1px solid black;padding:8px;text-align:left'>Hex</th></tr>
        <tr>{hex_row}</tr>
        <tr><th colspan="16" style='border:1px solid black;padding:8px;text-align:left'>ASCII</th></tr>
        <tr>{ascii_row}</tr>
    </table>
    """
    display(HTML(html))

In [11]:
plaintext = "strength"
key = "spirits"

state = text_to_state(plaintext)
key_state = text_to_state(key)

show_state(state, "Plaintext State")
show_state(key_state, "Key State")

cipher = aes_encrypt(state, key_state)

show_state(cipher, "Ciphertext")
show_cipher_hex_ascii(cipher)

0x73,0x6e,0x0,0x0
0x74,0x67,0x0,0x0
0x72,0x74,0x0,0x0
0x65,0x68,0x0,0x0


0x73,0x69,0x0,0x0
0x70,0x74,0x0,0x0
0x69,0x73,0x0,0x0
0x72,0x0,0x0,0x0


0x64,0xce,0x4f,0xd1
0xfc,0x62,0x35,0x3d
0x5d,0x61,0xa0,0x9
0x63,0x75,0x26,0x9b
